In [ ]:
from pathlib import Path

import pandas as pd

DATA_DIR = Path('data')


def read_data_csv(file_name, **kwargs):
    path = DATA_DIR / file_name
    if not path.exists():
        raise FileNotFoundError(f'Cannot find {path.resolve()}')
    return pd.read_csv(path, **kwargs)

In [ ]:
df_deposit = read_data_csv('Data_Deposit.csv')
print('Data_Deposit.csv loaded successfully.')
display(df_deposit.head())

In [ ]:
df_card = read_data_csv('Data_Card.csv')
print('Data_Card.csv loaded successfully.')
display(df_card.head())

In [ ]:
df_customer = read_data_csv('Data_Customer.csv', low_memory=False)
print('Data_Customer.csv loaded successfully.')
display(df_customer.head())

In [ ]:
df_activity = read_data_csv('Data_Activity.csv')
print('Data_Activity.csv loaded successfully.')
display(df_activity.head())

In [ ]:
df_transaction = read_data_csv('Data_Transaction.csv')
print('Data_Transaction.csv loaded successfully.')
display(df_transaction.head())

In [ ]:
df_lending = read_data_csv('Data_Lending.csv')
print('Data_Lending.csv loaded successfully.')
display(df_lending.head())

### Checking for Missing Values

In [ ]:
print('Missing values in df_deposit:')
display(df_deposit.isnull().sum())

In [ ]:
print('\nMissing values in df_card:')
display(df_card.isnull().sum())

In [ ]:
print('\nMissing values in df_customer:')
display(df_customer.isnull().sum())

cái này thì cứ thiếu sex là sẽ thiếu birth nên chỗ này em bỏ hết

---

4289 quan sát thiếu luôn

In [ ]:
# Drop rows where 'DATE_OF_BIRTH' is missing in df_customer
df_customer_cleaned = df_customer.dropna(subset=['DATE_OF_BIRTH']).copy()

print(f"Original df_customer shape: {df_customer.shape}")
print(f"Cleaned df_customer_cleaned shape: {df_customer_cleaned.shape}")
print('\nMissing values in df_customer:')
display(df_customer_cleaned.isnull().sum())

In [ ]:
print('\nMissing values in df_activity:')
display(df_activity.isnull().sum())

In [ ]:
print('\nMissing values in df_transaction:')
display(df_transaction.isnull().sum())

In [ ]:
print('\nMissing values in df_lending:')
display(df_lending.isnull().sum())

### Checking for Duplicated Rows

In [ ]:
print('Duplicated rows in df_deposit:', df_deposit.duplicated().sum())

In [ ]:
print('Duplicated rows in df_card:', df_card.duplicated().sum())

In [ ]:
print('Duplicated rows in df_customer_cleaned:', df_customer_cleaned.duplicated().sum())

In [ ]:
print('Duplicated rows in df_activity:', df_activity.duplicated().sum())

In [ ]:
print('Duplicated rows in df_transaction:', df_transaction.duplicated().sum())

In [ ]:
print('Duplicated rows in df_lending:', df_lending.duplicated().sum())

Không có file nào bị dính duplicated cả

### Checking for Outliers

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

def plot_numerical_outliers(df, df_name):
    numerical_cols = df.select_dtypes(include=['number']).columns
    customer_id_cols = [col for col in numerical_cols if 'CUSTOMER_NUMBER' in col.upper()]

    # Exclude customer ID columns from outlier plotting
    cols_to_plot = [col for col in numerical_cols if col not in customer_id_cols]

    if not cols_to_plot:
        print(f"No suitable numerical columns found for outlier detection in {df_name} (excluding customer IDs).")
        return

    print(f"\nOutliers for {df_name}:")

    for col in cols_to_plot:
        plt.figure(figsize=(8, 4))
        sns.boxplot(x=df[col])
        plt.title(f'Box Plot of {col} in {df_name}')
        plt.xlabel(col)
        plt.show()

# Check df_deposit
plot_numerical_outliers(df_deposit, 'df_deposit')

# Check df_card
plot_numerical_outliers(df_card, 'df_card')

# Check df_customer_cleaned
plot_numerical_outliers(df_customer_cleaned, 'df_customer_cleaned')

# Check df_activity
plot_numerical_outliers(df_activity, 'df_activity')

# Check df_transaction
plot_numerical_outliers(df_transaction, 'df_transaction')

# Check df_lending
plot_numerical_outliers(df_lending, 'df_lending')

Mấy giá trị Outlier bình thường, không cần phải sửa

### Investigating 'non-IB' Customers in Activity and Transaction Logs

In [ ]:
# Identify 'non-IB' customers from df_customer_cleaned
# Assuming 'non-IB' customers are those with a missing IB_REGISTER_DATE
non_ib_customers = df_customer_cleaned[df_customer_cleaned['IB_REGISTER_DATE'].isnull()]

# Get the CUSTOMER_NUMBER for these non-IB customers
non_ib_customer_numbers = non_ib_customers['CUSTOMER_NUMBER'].unique()

print(f"Number of identified 'non-IB' customers: {len(non_ib_customer_numbers)}")

In [ ]:
# Check if these non-IB customer numbers exist in df_activity
non_ib_in_activity = df_activity[df_activity['CUSTOMER_NUMBER'].isin(non_ib_customer_numbers)]

print(f"Number of 'non-IB' customers found in df_activity: {non_ib_in_activity['CUSTOMER_NUMBER'].nunique()}")
print(f"Total activity logs for 'non-IB' customers: {len(non_ib_in_activity)}")
display(non_ib_in_activity.head())

In [ ]:
# Check if these non-IB customer numbers exist in df_transaction
non_ib_in_transaction = df_transaction[df_transaction['CUSTOMER_NUMBER'].isin(non_ib_customer_numbers)]

print(f"Number of 'non-IB' customers found in df_transaction: {non_ib_in_transaction['CUSTOMER_NUMBER'].nunique()}")
print(f"Total transaction logs for 'non-IB' customers: {len(non_ib_in_transaction)}")
display(non_ib_in_transaction.head())

### Removing Inconsistent 'non-IB' Customer Data

In [ ]:
# Remove activity logs belonging to 'non-IB' customers
initial_activity_rows = len(df_activity)
df_activity_cleaned = df_activity[~df_activity['CUSTOMER_NUMBER'].isin(non_ib_customer_numbers)].copy()

print(f"Original df_activity rows: {initial_activity_rows}")
print(f"df_activity rows after removing non-IB entries: {len(df_activity_cleaned)}")
print(f"Number of activity rows removed: {initial_activity_rows - len(df_activity_cleaned)}")

display(df_activity_cleaned.head())

In [ ]:
# Remove transaction logs belonging to 'non-IB' customers
initial_transaction_rows = len(df_transaction)
df_transaction_cleaned = df_transaction[~df_transaction['CUSTOMER_NUMBER'].isin(non_ib_customer_numbers)].copy()

print(f"Original df_transaction rows: {initial_transaction_rows}")
print(f"df_transaction rows after removing non-IB entries: {len(df_transaction_cleaned)}")
print(f"Number of transaction rows removed: {initial_transaction_rows - len(df_transaction_cleaned)}")

display(df_transaction_cleaned.head())

The `df_activity_cleaned` and `df_transaction_cleaned` DataFrames now exclude the logs from the identified 'non-IB' customers. The original `df_activity` and `df_transaction` remain unchanged.

In [ ]:
from pathlib import Path

OUTPUT_DIR = Path('cleaned_data')
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)


def drop_unnamed_columns(df):
    cols = [c for c in df.columns if not str(c).startswith('Unnamed')]
    return df.loc[:, cols].copy()


def month_end(series):
    dt = pd.to_datetime(series, errors='coerce')
    return dt.dt.to_period('M').dt.to_timestamp('M')


def downcast_numeric(df):
    for col in df.select_dtypes(include=['int64', 'Int64']).columns:
        df[col] = pd.to_numeric(df[col], downcast='integer')
    for col in df.select_dtypes(include=['float64']).columns:
        df[col] = pd.to_numeric(df[col], downcast='float')
    return df


def save_table(df, name):
    parquet_path = OUTPUT_DIR / f'{name}.parquet'
    csv_path = OUTPUT_DIR / f'{name}.csv'
    try:
        df.to_parquet(parquet_path, index=False)
        print(f'Saved {parquet_path} shape={df.shape}')
    except Exception as exc:
        df.to_csv(csv_path, index=False)
        print(f'Parquet unavailable ({type(exc).__name__}); saved {csv_path} shape={df.shape}')


customer_clean = drop_unnamed_columns(df_customer_cleaned)
deposit_clean = drop_unnamed_columns(df_deposit)
lending_clean = drop_unnamed_columns(df_lending)
card_clean = drop_unnamed_columns(df_card)
activity_clean = drop_unnamed_columns(df_activity_cleaned)
transaction_clean = drop_unnamed_columns(df_transaction_cleaned)

for df in [deposit_clean, lending_clean, card_clean]:
    df['MONTH'] = month_end(df['MONTH'])

for col in ['CLIENT_CREATE_DATE', 'DATE_OF_BIRTH', 'IB_REGISTER_DATE']:
    if col in customer_clean.columns:
        customer_clean[col] = pd.to_datetime(customer_clean[col], errors='coerce')

activity_clean['ACTIVITY_DATE'] = pd.to_datetime(activity_clean['ACTIVITY_DATE'], errors='coerce')
activity_clean['MONTH'] = month_end(activity_clean['ACTIVITY_DATE'])

transaction_clean['TRANS_DATE'] = pd.to_datetime(transaction_clean['TRANS_DATE'], errors='coerce')
transaction_clean['MONTH'] = month_end(transaction_clean['TRANS_DATE'])
transaction_clean['IS_INTERNAL_TRANSFER'] = transaction_clean['Beneficiary_CUSTOMER_NUMBER'].notna().astype('int8')

for name, df in {
    'customer_clean': customer_clean,
    'deposit_clean': deposit_clean,
    'lending_clean': lending_clean,
    'card_clean': card_clean,
    'transaction_clean': transaction_clean,
    'activity_clean': activity_clean,
}.items():
    save_table(downcast_numeric(df), name)

transaction_monthly = (
    transaction_clean
    .groupby(['CUSTOMER_NUMBER', 'MONTH'], as_index=False)
    .agg(
        TRANS_RECORDS=('TRANS_NO', 'size'),
        TRANS_NO_SUM=('TRANS_NO', 'sum'),
        TRANS_AMOUNT_SUM=('TRANS_AMOUNT', 'sum'),
        TRANS_AMOUNT_MEAN=('TRANS_AMOUNT', 'mean'),
        TRANS_AMOUNT_MAX=('TRANS_AMOUNT', 'max'),
        TRANS_ACTIVE_DAYS=('TRANS_DATE', 'nunique'),
        TRANS_TYPE_LV1_NUNIQUE=('TRANS_LV1', 'nunique'),
        TRANS_TYPE_LV2_NUNIQUE=('TRANS_LV2', 'nunique'),
        DEVICE_NUNIQUE=('Device_ID_Hash', 'nunique'),
        MERCHANT_NUNIQUE=('Merchant_ID_Masked', 'nunique'),
        INTERNAL_TRANSFER_ROWS=('IS_INTERNAL_TRANSFER', 'sum'),
    )
)
transaction_monthly = downcast_numeric(transaction_monthly)
save_table(transaction_monthly, 'transaction_monthly_features')

activity_monthly = (
    activity_clean
    .groupby(['CUSTOMER_NUMBER', 'MONTH'], as_index=False)
    .agg(
        ACTIVITY_RECORDS=('ACTIVITY_NO', 'size'),
        ACTIVITY_NO_SUM=('ACTIVITY_NO', 'sum'),
        ACTIVITY_ACTIVE_DAYS=('ACTIVITY_DATE', 'nunique'),
        ACTIVITY_NAME_NUNIQUE=('ACTIVITY_NAME', 'nunique'),
        ACTIVITY_HOUR_NUNIQUE=('ACTIVITY_HOUR', 'nunique'),
    )
)
activity_monthly = downcast_numeric(activity_monthly)
save_table(activity_monthly, 'activity_monthly_features')

monthly_keys = pd.concat([
    deposit_clean[['CUSTOMER_NUMBER', 'MONTH']],
    lending_clean[['CUSTOMER_NUMBER', 'MONTH']],
    card_clean[['CUSTOMER_NUMBER', 'MONTH']],
    transaction_monthly[['CUSTOMER_NUMBER', 'MONTH']],
    activity_monthly[['CUSTOMER_NUMBER', 'MONTH']],
], ignore_index=True).drop_duplicates()

base = monthly_keys.merge(customer_clean, on='CUSTOMER_NUMBER', how='left')
base = base.merge(deposit_clean, on=['CUSTOMER_NUMBER', 'MONTH'], how='left')
base = base.merge(lending_clean, on=['CUSTOMER_NUMBER', 'MONTH'], how='left')
base = base.merge(card_clean, on=['CUSTOMER_NUMBER', 'MONTH'], how='left')
base = base.merge(transaction_monthly, on=['CUSTOMER_NUMBER', 'MONTH'], how='left')
base = base.merge(activity_monthly, on=['CUSTOMER_NUMBER', 'MONTH'], how='left')

fill_zero_prefixes = (
    'COUNT_', 'AVG_', 'OVERDUE_', 'LIMIT_', 'OUTSTANDING_', 'TERM_', 'INTEREST_',
    'TRANS_', 'ACTIVITY_', 'DEVICE_', 'MERCHANT_', 'INTERNAL_'
)
fill_zero_cols = [c for c in base.columns if c.startswith(fill_zero_prefixes)]
base[fill_zero_cols] = base[fill_zero_cols].fillna(0)

base = base.sort_values(['CUSTOMER_NUMBER', 'MONTH']).reset_index(drop=True)
base = downcast_numeric(base)

print(base.shape)
display(base.head())
save_table(base, 'gcon_customer_month_clean')

qa_df = pd.DataFrame([
    {
        'table': name,
        'rows': len(df),
        'cols': df.shape[1],
        'customer_nunique': df['CUSTOMER_NUMBER'].nunique() if 'CUSTOMER_NUMBER' in df.columns else pd.NA,
        'duplicate_rows': int(df.duplicated().sum()),
        'total_missing': int(df.isna().sum().sum()),
    }
    for name, df in {
        'customer_clean': customer_clean,
        'deposit_clean': deposit_clean,
        'lending_clean': lending_clean,
        'card_clean': card_clean,
        'transaction_clean': transaction_clean,
        'activity_clean': activity_clean,
        'gcon_customer_month_clean': base,
    }.items()
])
display(qa_df)
qa_df.to_csv(OUTPUT_DIR / 'cleaning_qa_summary.csv', index=False)